# Chapitre 1 — Lire les données · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch01_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre, de haut en bas : tant qu'une réponse manque, les cellules de vérification le signalent sans erreur. Le corrigé se trouve dans `ch01_corrige.ipynb`, sur la même page compagnon.

**Convention.** Comme dans le livre, la variance, l'écart-type, le score z et la covariance divisent par $n$ (`ddof=0`), sauf mention explicite.

**Sommaire.** Données · Les calculs des leçons · Exercices 1.1 à 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Couleurs du livre : une couleur = un rôle (observé, prédit, écart, droite du collègue)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Données

Les identifiants (D0.1, D1.1…) sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 1, recopiées des fichiers CSV du livre (ne pas modifier)

# D0.1 — les six logements de l'ouverture
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — prix de vente des T2 (k€), leçon 1.1
north = [174, 177, 181, 183, 185]
south  = [150, 165, 185, 195, 205]

# D1.2 — registre des ventes du quartier Sud, leçons 1.2 à 1.5
sales_log = pd.DataFrame({
    'sale':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'rooms':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'floor_area':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'energy_class':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'price':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'days_to_sell': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — prix au m² selon l'étage (k€/m²), leçon 1.5
floors = pd.DataFrame({
    'floor':        [0, 1, 2, 3, 4, 5, 6],
    'price_per_m2': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — délais de vente de trois agences (jours), exercice 1.1
agency_delays = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — loyers mensuels des studios (€), exercice 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

### Les variables des données

Les noms de code sont en anglais. Pour les colonnes, la dernière colonne donne le nom utilisé dans les fichiers CSV du livre.

| Nom | Sens | Unité | Colonne du CSV |
|:--|:--|:--|:--|
| `home` | identifiant du logement (A, B…) | — | `logement` |
| `floor_area` | surface habitable du logement | m² | `surface_m2` |
| `price` | prix de vente | k€ | `prix_kEUR` |
| `sale` | identifiant de la vente (S01…) | — | `vente` |
| `rooms` | nombre de pièces | pièces | `pieces` |
| `energy_class` | classe du diagnostic de performance énergétique (A à G) | — | `dpe` |
| `days_to_sell` | délai de vente | jours | `delai_jours` |
| `floor` | étage | — | `etage` |
| `price_per_m2` | prix au mètre carré | k€/m² | `prix_m2_kEUR` |

In [ ]:
# Outils fournis : vous n'avez pas à les modifier

def book_quartiles(values):
    """Q1 et Q3 selon la règle du livre (équation 1.5) : rangs ⌈n/4⌉ et ⌈3n/4⌉."""
    x = sorted(values)
    n = len(x)
    rank_q1 = -(-n // 4)          # arrondi à l'entier supérieur de n/4
    rank_q3 = -(-3 * n // 4)      # arrondi à l'entier supérieur de 3n/4
    return x[rank_q1 - 1], x[rank_q3 - 1]   # rang 1 = position 0 en Python


def dot_rows(series, xlabel, bands=None, ax=None):
    """Une rangée de points par série, comme les figures de la leçon 1.1.
    bands : dictionnaire {nom: (moyenne, écart-type)} pour dessiner moyenne ± écart-type."""
    if ax is None:
        fig, ax = plt.subplots()
    names = list(series)
    colors = [BLUE, ORANGE, '#52514e']
    for i, name in enumerate(names):
        y = len(names) - 1 - i
        if bands and name in bands:
            m, s = bands[name]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[name], [y] * len(series[name]), s=50,
                   color=colors[i % 3], zorder=3)
    ax.set_yticks(range(len(names)), list(reversed(names)))
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 1.1 à 1.5. Exécutez-les et comparez avec le livre. Ici, `np.std` et `np.var` sont appelés avec `ddof=0`, c'est-à-dire avec une division par $n$, comme dans le chapitre.

In [ ]:
# Leçon 1.1 — moyenne, variance et écart-type (division par n)
for name, one_series in [('Nord', north), ('Sud', south)]:
    print(f'{name} : moyenne = {np.mean(one_series):g} k€, variance = {np.var(one_series, ddof=0):g} k€², '
          f'écart-type = {np.std(one_series, ddof=0):g} k€')

In [ ]:
# Leçon 1.2 — effectifs par classe DPE (variable qualitative ordinale)
sales_log['energy_class'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Leçon 1.3 — moyenne, médiane et quartiles des délais
d = sales_log['days_to_sell']
print('moyenne :', d.mean(), ' médiane :', d.median())
print('sans S13 : moyenne', d[sales_log['sale'] != 'S13'].mean(),
      ' médiane', d[sales_log['sale'] != 'S13'].median())
print('quartiles, règle du livre (équation 1.5) :', book_quartiles(d))
print('quartiles, NumPy par défaut (interpolation) :', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Leçon 1.4 — scores standardisés (équation 1.6, écart-type avec division par n)
for name, one_series, sale in [('Nord', north, 188), ('Sud', south, 210)]:
    m, s = np.mean(one_series), np.std(one_series, ddof=0)
    print(f'{name} : z de la série = {np.round((np.array(one_series) - m) / s, 4)}, '
          f'z de la vente à {sale} k€ = {(sale - m) / s:g}')

In [ ]:
# Leçon 1.5 — covariance et corrélation des cinq T2 du Sud (division par n)
t2 = sales_log[sales_log['rooms'] == 2]
x, y = t2['floor_area'].to_numpy(float), t2['price'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covariance :', cov, 'm²·k€   r :', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r par np.corrcoef :', np.corrcoef(x, y)[0, 1].round(4))
print('r étage / prix au m² :', np.corrcoef(floors['floor'], floors['price_per_m2'])[0, 1].round(4))

## Exercice 1.1 — Lire un graphique

*Données D1.E1. Notions : moyenne et écart-type (leçon 1.1).* Le graphique ci-dessous reproduit celui de l'exercice 1.1 du livre.

In [ ]:
dot_rows({f'Agence {a}': v for a, v in agency_delays.items()},
                  'Délai de vente (jours)')
plt.xlim(0, 60)
plt.show()

**a) et b)** Estimez à l'œil le délai moyen de chaque agence, puis rangez les agences de la moins dispersée à la plus dispersée.

In [ ]:
# À COMPLÉTER : vos estimations à l'œil, avant tout calcul
estimated_mean = {'P': None, 'Q': None, 'R': None}   # en jours
spread_order = None   # liste, de la moins à la plus dispersée, ex. ['Q', 'P', 'R']

In [ ]:
# Vérification (division par n)
for a, v in agency_delays.items():
    print(f'Agence {a} : moyenne = {np.mean(v):g} j, écart-type = {np.std(v, ddof=0):.2f} j')
true_order = sorted(agency_delays, key=lambda a: np.std(agency_delays[a]))
if spread_order is None or None in estimated_mean.values():
    print('Complétez la cellule précédente.')
else:
    print('Ordre de dispersion :', 'correct' if spread_order == true_order else 'à revoir')

**c)** Comptez, pour Q et pour R, les ventes conclues à 2 jours ou moins de la moyenne.

In [ ]:
# À COMPLÉTER : vos comptages lus sur le graphique
close_to_Q = None
close_to_R = None

In [ ]:
# Vérification
for a, yours in [('Q', close_to_Q), ('R', close_to_R)]:
    v = np.array(agency_delays[a])
    truth = int(np.sum(np.abs(v - v.mean()) <= 2))
    if yours is None:
        print(f'Agence {a} : pas encore complété.')
    else:
        print(f"Agence {a} : {'correct' if yours == truth else 'à revoir'}")

*Votre explication pour c) :* …

## Exercice 1.2 — Deux résidences, un même loyer moyen

*Données D1.E2. Notions : moyenne, variance et écart-type (leçon 1.1).* Camille ne peut pas dépasser 570 € par mois. Les questions a) à d) se font d'abord sur papier, avec le tableau de l'exercice 1.2 du livre. Ce notebook sert à vérifier vos réponses, puis à programmer l'écart-type (question e) et à comparer deux conventions (question f).

In [ ]:
dot_rows({'Les Tilleuls': tilleuls, 'Le Port': port}, "Loyer mensuel d'un studio (€)")
plt.axvline(570, color='black', linestyle=':', label='budget de Camille : 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Lire les loyers

Combien de studios Camille peut-elle louer dans chaque résidence avec 570 €, puis avec 620 € ?

In [ ]:
# À COMPLÉTER : nombre de studios accessibles, lu dans la liste des loyers
affordable = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Vérification : `<=` produit des vrai/faux, np.sum compte les « vrai »
for budget, answers in affordable.items():
    for name, one_series in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        truth = int(np.sum(np.array(one_series) <= budget))
        yours = answers[name]
        status = 'pas encore complété' if yours is None else ('correct' if yours == truth else 'à revoir')
        print(f'Budget {budget} €, {name} : {status}')

*Votre réponse à a), en une phrase :* …

### b) Les deux moyennes

Calculez-les sur papier, puis reportez vos résultats ci-dessous.

In [ ]:
# À COMPLÉTER : vos moyennes calculées à la main (en €)
mean_tilleuls_by_hand = None
mean_port_by_hand = None

In [ ]:
# Vérification
for name, one_series, yours in [('Les Tilleuls', tilleuls, mean_tilleuls_by_hand),
                              ('Le Port', port, mean_port_by_hand)]:
    if yours is None:
        print(f'{name} : complétez la cellule précédente.')
    else:
        ok = np.isclose(yours, np.mean(one_series))
        print(f"{name} : {'correct' if ok else 'à revoir'} (votre valeur : {yours} €)")

### c) Variance et écart-type à la main (division par $n = 6$)

Sur papier, dressez pour chaque résidence le tableau des écarts à la moyenne et de leurs carrés (modèle : le tableau du calcul pas à pas de la leçon 1.1). Reportez ici les quatre résultats.

In [ ]:
# À COMPLÉTER : vos résultats calculés à la main (division par n)
variance_tilleuls_by_hand = None   # en €²
variance_port_by_hand = None       # en €²
std_tilleuls_by_hand = None # en €
std_port_by_hand = None     # en €

In [ ]:
# Vérification (division par n : ddof=0)
your_values = {
    'variance Tilleuls': (variance_tilleuls_by_hand, np.var(tilleuls, ddof=0)),
    'variance Port': (variance_port_by_hand, np.var(port, ddof=0)),
    'écart-type Tilleuls': (std_tilleuls_by_hand, np.std(tilleuls, ddof=0)),
    'écart-type Port': (std_port_by_hand, np.std(port, ddof=0)),
}
for name, (yours, expected) in your_values.items():
    if yours is None:
        print(f'{name} : pas encore complété.')
    else:
        print(f"{name} : {'correct' if np.isclose(yours, expected) else 'à revoir'}")

### d) Interpréter

*Votre réponse :* …

### e) Programmer l'écart-type

Complétez la fonction en suivant l'équation 1.3 du livre, avec une division par $n$ :

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

N'utilisez ni `np.std` ni `np.var` : le but est d'écrire chaque étape vous-même.

In [ ]:
# Exercice 1.2 e) — À COMPLÉTER
def std_dev(values):
    """Écart-type, division par n (équation 1.3)."""
    x = np.array(values, dtype=float)
    n = len(x)
    mean = None   # À COMPLÉTER : équation 1.1
    deviations = None    # À COMPLÉTER : un écart par valeur
    variance = None  # À COMPLÉTER : moyenne des carrés
    return None      # À COMPLÉTER : retour à l'unité
ecart_type = std_dev   # alias du livre (nom de l'édition française imprimée)

<details><summary><b>Indice</b> (cliquez si vous bloquez)</summary>

Avec NumPy, une opération écrite sur un tableau s'applique à chaque valeur : `x - mean` donne d'un coup les six écarts, et `deviations ** 2` leurs six carrés. `np.sum(...)` additionne, `np.sqrt(...)` prend la racine carrée.

</details>

In [ ]:
# Vérification : exécutez cette cellule sans la modifier
if std_dev(tilleuls) is None:
    print('La fonction renvoie encore None : complétez-la.')
else:
    for name, one_series, expected in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtained = std_dev(one_series)
        status = 'correct' if np.isclose(obtained, expected) else f'à revoir (attendu : {expected})'
        print(f'{name} : {obtained:.2f} € → {status}')

### f) Approfondissement : diviser par $n$ ou par $n - 1$

Exécutez la cellule ci-dessous. Retrouvez 10,95 à la main en divisant la somme des carrés par $n - 1 = 5$. Quelle option de pandas donne la convention du chapitre ?

*Votre réponse :* …

In [ ]:
rents = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, par défaut (division par n − 1) :', rents.std().round(2).to_dict())
print('pandas, ddof=0 (division par n)          :', rents.std(ddof=0).round(2).to_dict())

## Exercice 1.3 — Une vente surprenante ?

*Données D1.1. Notions : score standardisé (leçon 1.4, équation 1.6).* Un T2 vient de se vendre 196 k€. Moyennes et écarts-types : ceux de la leçon 1.1, avec division par $n$.

In [ ]:
# À COMPLÉTER : vos calculs faits sur papier
z_north = None
z_south = None
south_price_as_unusual = None   # en k€, question c)

In [ ]:
# Vérification (écart-type avec division par n)
expected_values = {
    'z Nord': (z_north, (196 - np.mean(north)) / np.std(north, ddof=0)),
    'z Sud': (z_south, (196 - np.mean(south)) / np.std(south, ddof=0)),
    'prix Sud équivalent': (south_price_as_unusual,
                            np.mean(south) + (196 - np.mean(north)) / np.std(north, ddof=0) * np.std(south, ddof=0)),
}
for name, (yours, truth) in expected_values.items():
    status = 'pas encore complété' if yours is None else ('correct' if np.isclose(yours, truth) else 'à revoir')
    print(f'{name} : {status}')

*Votre réponse à b) :* …

## Exercice 1.4 — Surface et prix vont-ils ensemble ?

*Données D0.1 (les six logements de l'ouverture). Notions : covariance et corrélation (leçon 1.5, équations 1.7 et 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(homes['floor_area'], homes['price'], color=BLUE)
for _, l in homes.iterrows():
    plt.annotate(l['home'], (l['floor_area'], l['price']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Surface (m²)')
plt.ylabel('Prix de vente (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Avant tout calcul : $r$ sera-t-il positif ou négatif ? Proche de 0, de 0,5 ou de 1 ?

*Votre réponse :* …

**b)** Complétez les lignes ci-dessous. Utilisez une division par $n$ partout.

In [ ]:
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
n = len(x)

products = None     # À COMPLÉTER : (x_i − x̄)(y_i − ȳ) pour chaque logement
produits = products   # alias du livre (nom de l'édition française imprimée)
covariance = None   # À COMPLÉTER : équation 1.7, division par n
r = None            # À COMPLÉTER : équation 1.8 (écarts-types avec ddof=0)

<details><summary><b>Indice</b></summary>

`x.mean()` donne la moyenne, `x - x.mean()` les six écarts. Le produit `*` de deux tableaux multiplie terme à terme. `x.std(ddof=0)` donne l'écart-type avec division par $n$.

</details>

In [ ]:
# Vérification
if r is None or covariance is None:
    print('Complétez la cellule précédente.')
else:
    print('covariance :', round(covariance, 2), 'm²·k€ →',
          'correct' if np.isclose(covariance, 1637.5) else 'à revoir (attendu : 1637,5)')
    print('r :', round(r, 4), '→', 'correct' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'à revoir')

**c)** Corrélation proche de 1, mais écarts de 5 à 11 k€ en valeur absolue : est-ce contradictoire ?

*Votre réponse :* …

In [ ]:
print('np.cov par défaut (division par n − 1) :', np.cov(x, y)[0, 1])
print('np.cov avec bias=True (division par n)  :', np.cov(x, y, bias=True)[0, 1])

## Exercice 1.5 — Prédire, puis vérifier

*Données D1.1. Notions : moyenne et écart-type (leçon 1.1), médiane et écart interquartile (leçon 1.3).* Un sixième T2 se vend au Nord pour 240 k€.

**a)** Avant tout calcul, notez vos prédictions : la moyenne, la médiane, l'écart-type et l'écart interquartile augmentent-ils beaucoup, un peu ou pas du tout ?

*Vos prédictions :* …

In [ ]:
north_plus = north + [240]
print('Nord :', sorted(north))
print('Nord avec la vente à 240 k€ :', sorted(north_plus))

**b)** Complétez la fonction `summary`. Elle doit renvoyer les quatre indicateurs, avec l'écart-type divisé par $n$ et les quartiles calculés par la règle du livre (`book_quartiles`, fournie plus haut).

In [ ]:
# À COMPLÉTER
def summary(one_series):
    q1, q3 = book_quartiles(one_series)
    return {
        'moyenne': None,              # À COMPLÉTER
        'médiane': None,              # À COMPLÉTER : np.median
        'écart-type': None,           # À COMPLÉTER : division par n
        'écart interquartile': None,  # À COMPLÉTER : à partir de q1 et q3
    }
resume = summary   # alias du livre (nom de l'édition française imprimée)

In [ ]:
table = pd.DataFrame({'Nord (5 ventes)': summary(north), 'Nord + 240 k€ (6 ventes)': summary(north_plus),
                        'Sud (5 ventes)': summary(south)})
if table.isna().any().any():
    print('Complétez la fonction summary.')
else:
    expected = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (before, after) in expected.items():
        ok = np.isclose(table.loc[k].iloc[0], before) and np.isclose(table.loc[k].iloc[1], after, atol=0.005)
        print(f"{k} : {'correct' if ok else 'à revoir'}")
table.round(2)

In [ ]:
# Les deux séries sur le modèle de la figure « moyenne ± écart-type » de la leçon 1.1
if table.isna().any().any():
    print('Complétez la fonction summary pour afficher le graphique.')
else:
    dot_rows({'Nord': north, 'Nord + 240': north_plus},
                      'Prix de vente d\'un T2 (k€)',
                      bands={'Nord': (np.mean(north), np.std(north, ddof=0)),
                              'Nord + 240': (np.mean(north_plus), np.std(north_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** Les prix du Nord sont-ils devenus plus imprévisibles que ceux du Sud ?

*Votre réponse :* …

## Exercice 1.6 — Défi : deux droites, une même erreur moyenne

*Données D0.1. Notions : écart signé et valeur absolue (ouverture, leçon 1.1), écart-type (leçon 1.1).* Droite tracée à l'œil : prix prédit = 3 × surface + 30. Droite paresseuse : prix prédit = 222 k€ pour tous les logements.

In [ ]:
price = homes['price'].to_numpy(dtype=float)
floor_area = homes['floor_area'].to_numpy(dtype=float)

eye_line_errors = None          # À COMPLÉTER : observé − prédit, droite tracée à l'œil
lazy_line_errors = None    # À COMPLÉTER : observé − prédit, droite paresseuse

In [ ]:
# Les trois indicateurs de la question b) (écart-type : division par n)
if eye_line_errors is None or lazy_line_errors is None:
    print('Complétez la cellule précédente.')
else:
    indicators = pd.DataFrame({
        name: {'moyenne des écarts signés': e.mean(),
              'moyenne des écarts en valeur absolue': np.abs(e).mean(),
              'écart-type des écarts signés': e.std(ddof=0)}
        for name, e in [('droite à l\'œil', eye_line_errors), ('droite paresseuse', lazy_line_errors)]
    })
    print('écarts, droite à l\'œil   :', eye_line_errors)
    print('écarts, droite paresseuse :', lazy_line_errors)
    print('vérification :', 'correct' if np.allclose(indicators.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicators.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'à revoir')
    display(indicators.round(2))

In [ ]:
if eye_line_errors is not None and lazy_line_errors is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(floor_area - 0.8, eye_line_errors, color=ORANGE, label="droite à l'œil")
    ax.scatter(floor_area + 0.8, lazy_line_errors, color=BLUE, marker='s', label='droite paresseuse')
    ax.set_xlabel('Surface (m²)')
    ax.set_ylabel('Écart observé − prédit (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Vos réponses à c) et d) :* …

In [ ]:
# Contrôle de d) : l'écart-type des écarts de la droite paresseuse égale celui des prix
if lazy_line_errors is not None:
    print(round(lazy_line_errors.std(ddof=0), 4), round(price.std(ddof=0), 4))